# Notebook09 exploratory four-arm evaluation definitions

Reusable definitions only. Inputs: all eight completed350-update arms, immutable initial/final teachers, fixed10,100probe-training and2,020development identities. Outputs: sixteen fixedepoch20probe results, individual paired effects and fragile t intervals(df1), own-initialization gains, local-versus-MLP and far-versus-local controls, representation diagnostics, counts and descriptive compute figures. No confirmatory primary endpoint or winner-selection rule exists. Official test remains reserved.

Runtime is reserved inside the eventual three-hour study clock. Exact resume: Run All in `notebooks/09_Deep_Energy_Encoder_Pilot.ipynb` with the same run ID and unchanged sources; cell `launch-deep-energy`. Probe checkpoints:`checkpoints/<run_id>/stages/learning/probes/`. BUILD ONLY: importing never launches training or evaluation. Both seeds20261201/20261202 and all16probe jobs are mandatory; incomplete results cannot become an architectural conclusion. Old07cached features are allowed only in the clearly marked software-validation smoke, never in09quality results.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT=Path(os.environ.get("FMM_JEPA_ROOT","/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot={"__file__":str(ROOT/"fmm_jepa/bootstrap.ipynb")}
for _cell in nbformat.read(ROOT/"fmm_jepa/bootstrap.ipynb",as_version=4).cells:
    if _cell.cell_type=="code" and "ahr-export" in _cell.metadata.get("tags",[]):
        exec(compile(_cell.source,"bootstrap#ahr-export","exec"),_boot)
_boot["install_notebook_package"](ROOT)

"""Notebook09 fixed epoch20 probes and paired exploratory evidence."""
from pathlib import Path
import copy, hashlib, json, math, sys, time
import numpy as np
import torch

ROOT=Path(__file__).resolve().parents[2]
if 'fmm09_runner' not in sys.modules:
    import types
    _path=ROOT/'experiments/09_deep_energy/runner.ipynb'
    _module=types.ModuleType('fmm09_runner');_module.__file__=str(_path);sys.modules['fmm09_runner']=_module
    for _cell in json.loads(_path.read_text())['cells']:
        if _cell['cell_type']=='code' and 'ahr-export' in _cell.get('metadata',{}).get('tags',[]):
            exec(compile(''.join(_cell['source']),str(_path),'exec'),_module.__dict__)
RUNNER=sys.modules['fmm09_runner']
_BASE=RUNNER.load_tagged(ROOT,'experiments/05_scaling/evaluation.ipynb','fmm09_inherited_evaluation')
for _name,_value in list(_BASE.__dict__.items()):
    if not _name.startswith('__') and _name not in globals():globals()[_name]=_value
build_model=RUNNER.build_model


def _evaluation_export_hash(root):
    return _digest([''.join(c['source']) for c in _read(Path(root)/'experiments/09_deep_energy/evaluation.ipynb')['cells']
        if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])])

def _verify_plan_sources(root,plan):
    actual=RUNNER.stage_definition_hashes(root)
    if plan['source_hashes']!=actual:raise ValueError('Notebook09 registered definitions changed')
    if plan.get('consumer_notebook_sha256'):
        code=[''.join(c['source']) for c in _read(Path(root)/plan['resume_notebook'])['cells'] if c['cell_type']=='code']
        if _digest(code)!=plan['consumer_notebook_sha256']:raise ValueError('Registered main notebook code changed')
    return actual

def _stage_jobs(plan,training):
    if training.get('status')!='COMPLETE' or not training.get('equal_updates') or not training.get('paired_traces_within_seed') or not training.get('no_failed_attempts'):
        raise ValueError('Only clean, fully paired, equal-update training may enter quality evaluation')
    if set(training['arms'])!={a['id'] for a in plan['arms']}:raise ValueError('Training arm identities differ')
    jobs=[]
    for arm in plan['arms']:
        if arm['config']['training']['updates']!=plan['updates'] or arm['config']['probe']['epochs']!=20:
            raise ValueError('Schedule/probe budget differs')
        if arm['config']['evaluation'].get('heldout_enabled',False):raise ValueError('Official test remains reserved')
        record=training['arms'][arm['id']]
        if record['successful_updates']!=plan['updates'] or record['attempted_updates']!=plan['updates'] or record['failures']:
            raise ValueError('Incomplete or numerically failed arm')
        jobs.append({'model_id':arm['id']+'-initial','arm':arm,'updates':0,
            'control':'initialization','state':record['initial_teacher'],'training_cost':None})
        for milestone in plan['milestones']:
            state=record['milestones'][str(milestone)]
            if state['successful_updates']!=milestone:raise ValueError('Milestone update differs')
            jobs.append({'model_id':arm['id']+'-u'+str(milestone),'arm':arm,'updates':milestone,
                'control':'trained','state':state,'training_cost':state})
    return jobs

def _seed_statistics(rows,value_key='delta_percentage_points',expected_n=None):
    from scipy.stats import t
    n=len(rows)
    if n<2 or len({int(row['seed']) for row in rows})!=n or (expected_n is not None and n!=int(expected_n)):
        raise ValueError('Distinct paired training seeds and the registered sample count are required')
    values=np.asarray([float(row[value_key]) for row in rows],dtype=np.float64)
    if not np.isfinite(values).all():raise ValueError('Nonfinite paired seed effect')
    mean=float(values.mean());sd=float(values.std(ddof=1));se=sd/math.sqrt(n);df=n-1
    half=float(t.ppf(.975,df))*se
    statistic=mean/se if se>0 else 0. if mean==0 else None
    p=float(2*t.sf(abs(statistic),df)) if statistic is not None else 0.
    return {'n_training_seeds':n,'degrees_of_freedom':df,'mean_percentage_points':mean,
        'sd_percentage_points':sd,'standard_error_percentage_points':se,
        'approximate_t_ci95_percentage_points':[mean-half,mean+half],
        't_statistic':statistic,'exploratory_two_sided_p_value':p,'zero_variance_deltas':se==0,
        'range_percentage_points':[float(values.min()),float(values.max())],'positive_seeds':int((values>0).sum()),
        'paired_seed_deltas':[{'seed':int(row['seed']),'delta_percentage_points':float(row[value_key])} for row in rows],
        'uncertainty':'Exploratory paired training-seed t interval; two pairs implydf1 and very uncertain variance. Conditional on reused development images; no confirmatory endpoint, no multiplicity adjustment, no selected winner.'}

def analyze_stage_evidence(evaluations):
    index={(value['variant'],int(value['seed']),int(value['updates'])):value for value in evaluations.values()}
    seeds=sorted({key[1] for key in index});variants=['vit','mlp','local','local_far'];steps=[0,350]
    if seeds!=[20261201,20261202] or {key[0] for key in index}!=set(variants) or {key[2] for key in index}!=set(steps):
        raise ValueError('Expected two fresh paired seeds, four architectures and initial/final350updates')
    if len(index)!=len(evaluations) or len(index)!=16:raise ValueError('All16 unique initial/final probe results are required')
    contrasts={};rows=[]
    def paired(name,values):
        contrasts[name]=_seed_statistics([{'seed':seed,'delta_percentage_points':value} for seed,value in zip(seeds,values)],expected_n=2)
    for variant in variants:
        paired(variant+'_learning_u350',[_paired_delta(index[(variant,seed,350)],index[(variant,seed,0)]) for seed in seeds])
    comparisons=[('mlp','vit'),('local','vit'),('local_far','vit'),('local','mlp'),('local_far','local')]
    for candidate,control in comparisons:
        for step in steps:
            paired(candidate+'_minus_'+control+'_u'+str(step),[_paired_delta(index[(candidate,seed,step)],index[(control,seed,step)]) for seed in seeds])
        paired(candidate+'_minus_'+control+'_learning_gain_u350',[
            _paired_delta(index[(candidate,seed,350)],index[(candidate,seed,0)])-_paired_delta(index[(control,seed,350)],index[(control,seed,0)]) for seed in seeds])
    for variant in variants:
        for seed in seeds:
            initial=_score(index[(variant,seed,0)])
            for step in steps:
                saved=index[(variant,seed,step)];cost=saved.get('training_cost')or{};perf=cost.get('performance')or{}
                peak=perf.get('peak_cuda_allocated_bytes');diagnostic=saved['representation_diagnostics']['descriptive']
                rows.append({'model_id':saved['model_id'],'variant':variant,'kind':saved['kind'],'seed':seed,'updates':step,
                    'top1_percent':_score(saved),'initial_top1_percent':initial,'gain_percentage_points':_score(saved)-initial,
                    'image_exposures':int(cost.get('image_exposures',0)),
                    'cumulative_training_step_seconds':cost.get('cumulative_step_seconds')if step else 0.,
                    'median_training_step_seconds':(perf.get('complete_jepa_step_seconds')or{}).get('median'),
                    'peak_training_GiB':float(peak)/1024**3 if peak is not None else None,
                    'encoder_parameters':saved['parameter_counts']['context'],'trainable_parameters':saved['parameter_counts']['trainable'],
                    'inactive_context_scaffold_parameters':saved['parameter_counts'].get('inactive_context_scaffold',0),
                    'encoder_active_parameters':saved['parameter_counts']['context_active'],
                    'trainable_active_parameters':saved['parameter_counts']['trainable_active'],
                    'feature_std_mean':diagnostic['feature_std_mean'],'effective_rank':diagnostic['effective_rank']})
    sanity={variant:{str(seed):{'finite':bool(index[(variant,seed,350)]['representation_diagnostics']['finite_train_and_development']),
        'not_exactly_constant':bool(index[(variant,seed,350)]['representation_diagnostics']['development_representation_not_exactly_constant'] and index[(variant,seed,350)]['representation_diagnostics']['training_representation_not_exactly_constant'])} for seed in seeds} for variant in variants}
    learning={variant:{'mean_gain_percentage_points':contrasts[variant+'_learning_u350']['mean_percentage_points'],
        'interval_excludes_zero_positive':contrasts[variant+'_learning_u350']['approximate_t_ci95_percentage_points'][0]>0,
        'interval_excludes_zero_negative':contrasts[variant+'_learning_u350']['approximate_t_ci95_percentage_points'][1]<0} for variant in variants}
    gate={'passed':all(row['finite'] and row['not_exactly_constant'] for group in sanity.values() for row in group.values()),
        'status':'NUMERICAL_SANITY_ONLY','representation_checks':sanity,'own_learning':learning,
        'reason':'Finite nonconstant features are numerical checks. All quality contrasts are exploratory; no automatic continuation or winner selection.'}
    assessment={'status':'EXPLORATORY_RESULTS_COMPLETE','registered_pairs':2,'degrees_of_freedom':1,
        'confirmatory_primary_endpoint':None,'mechanism_comparisons':['local_minus_mlp_u350','local_far_minus_local_u350'],
        'interpretation':'Report both seed differences, fragile paired t intervals, own-initialization gains and measured costs. Two seeds do not establish a general architectural winner.'}
    return {'contrasts':contrasts,'chart_rows':rows,'learning_gate':gate,'hypothesis_assessment':assessment}

def write_stage_figures(artifact,result,summary):
    """Show the two observed seed effects explicitly; no precision shading."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    variants=['vit','mlp','local','local_far'];labels=['ViT','MLP only','Local energy','Local + far energy']
    colors=['#59636E','#B17E3D','#4B77A8','#697C53'];markers=['o','s','^','D'];steps=[0,350]
    fig,axes=plt.subplots(1,2,figsize=(13,4.8),layout='constrained')
    for i,variant in enumerate(variants):
        rows=[row for row in result['chart_rows'] if row['variant']==variant]
        for seed_index,seed in enumerate([20261201,20261202]):
            values=sorted([row for row in rows if row['seed']==seed],key=lambda row:row['updates'])
            axes[0].plot([row['updates'] for row in values],[row['top1_percent'] for row in values],color=colors[i],
                marker=markers[i],linestyle='-'if seed_index==0 else'--',alpha=.8,label=labels[i]if seed_index==0 else None)
        effects=result['contrasts'][variant+'_learning_u350']['paired_seed_deltas']
        for offset,row in zip([-.1,.1],effects):axes[1].scatter(row['delta_percentage_points'],i+offset,color=colors[i],marker=markers[i],s=50)
        axes[1].scatter(result['contrasts'][variant+'_learning_u350']['mean_percentage_points'],i,color='#222222',marker='|',s=100)
    axes[0].set(title='Each seed: development accuracy before and after training',xlabel='Successful training updates',ylabel='Top1 accuracy (%)',xticks=steps)
    axes[0].set_ylim(bottom=0);axes[0].legend(frameon=False)
    axes[1].axvline(0,color='#59636E',linewidth=1);axes[1].set(title='Own-initialization learning: two seed effects and mean',xlabel='Gain (percentage points)',yticks=range(4),yticklabels=labels)
    for axis in axes:axis.spines[['top','right']].set_visible(False);axis.grid(axis='x',alpha=.15)
    fig.suptitle('Food101 reused development images; fixed epoch20 probes; exploratory two-seed pilot',fontsize=12)
    quality=Path(artifact)/'stage_quality.png';fig.savefig(quality,dpi=160);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(12,4.6),layout='constrained')
    for axis,key,title,unit in ((axes[0],'step_seconds','Training step time','Seconds'),(axes[1],'peak_GiB','Peak allocated GPU memory','GiB')):
        values=[row[key] for row in summary]
        if any(value is None for value in values):axis.text(.5,.5,'UNKNOWN: incomplete measurements',ha='center',transform=axis.transAxes)
        else:
            axis.bar(range(4),values,color=colors,edgecolor='#333333',linewidth=.5)
            for i,value in enumerate(values):axis.annotate(f'{value:.3f}',(i,value),xytext=(0,4),textcoords='offset points',ha='center')
            axis.set_ylim(0,max(values)*1.18)
        axis.set(title=title,ylabel=unit,xticks=range(4),xticklabels=['ViT','MLP','Local','Local + far']);axis.spines[['top','right']].set_visible(False)
    fig.suptitle('Means of per-seed measurements; equal exposure, different compute',fontsize=12)
    compute=Path(artifact)/'stage_compute.png';fig.savefig(compute,dpi=160);plt.close(fig)
    comparisons=['local_minus_mlp_u350','local_far_minus_local_u350'];names=['Local energy minus MLP','Far contribution: local + far minus local']
    fig,axis=plt.subplots(figsize=(10,3.8),layout='constrained')
    for i,key in enumerate(comparisons):
        stats=result['contrasts'][key]
        for offset,row in zip([-.1,.1],stats['paired_seed_deltas']):axis.scatter(row['delta_percentage_points'],i+offset,color='#4B77A8',marker='o',s=55)
        axis.scatter(stats['mean_percentage_points'],i,color='#222222',marker='|',s=140)
    axis.axvline(0,color='#59636E',linewidth=1);axis.set(yticks=range(2),yticklabels=names,xlabel='Final development accuracy difference (percentage points)',title='Exploratory mechanism controls: each seed and mean')
    axis.spines[['top','right']].set_visible(False);axis.grid(axis='x',alpha=.15)
    mechanism=Path(artifact)/'stage_contrasts.png';fig.savefig(mechanism,dpi=160);plt.close(fig)
    records={name:{'path':str(path.resolve()),'sha256':digest_file(path)} for name,path in [('quality',quality),('compute',compute),('mechanism_controls',mechanism)]}
    records['visual_inspection']='Pending inspection after real experiment completion; no fabricated results during build.'
    atomic_json(Path(artifact)/'stage_figure_manifest.json',records)
    return records

def write_stage_report(artifact_dir,result):
    artifact=Path(artifact_dir);final=350;rows=[row for row in result['chart_rows'] if row['updates']==final]
    variants=['vit','mlp','local','local_far'];summary=[]
    for variant in variants:
        group=[row for row in rows if row['variant']==variant]
        def mean(name):return float(np.mean([row[name] for row in group]))if all(row[name]is not None for row in group)else None
        summary.append({'variant':variant,'top1_percent':mean('top1_percent'),'gain_percentage_points':mean('gain_percentage_points'),
            'step_seconds':mean('median_training_step_seconds'),'peak_GiB':mean('peak_training_GiB'),
            'encoder_parameters':mean('encoder_parameters'),'trainable_parameters':mean('trainable_parameters'),
            'inactive_context_scaffold_parameters':mean('inactive_context_scaffold_parameters'),
            'encoder_active_parameters':mean('encoder_active_parameters'),'trainable_active_parameters':mean('trainable_active_parameters'),
            'feature_std_mean':mean('feature_std_mean'),'effective_rank':mean('effective_rank')})
    indexed={row['variant']:row for row in summary};ratios={}
    for candidate,control in [('mlp','vit'),('local','vit'),('local_far','vit'),('local','mlp'),('local_far','local')]:
        ratios[candidate+'_over_'+control]={}
        for field in ('step_seconds','peak_GiB'):
            a=indexed[candidate][field];b=indexed[control][field]
            ratios[candidate+'_over_'+control][field]=a/b if a is not None and b is not None and b>0 else None
    atomic_json(artifact/'stage_cost_comparisons.json',ratios)
    figures=write_stage_figures(artifact,result,summary)
    lines=['# Notebook09 deep energy encoder: exploratory four-arm pilot','',
        'Status: COMPLETE. Both fresh seeds completed all four architectures at350updates and all16initial/final fixedepoch20 probes. Official test: NOT_RUN.',
        '', 'This is exploratory evidence on reused development images. No confirmatory primary endpoint, statistical winner selection or automatic continuation was registered.',
        '', '| Variant | Development top1 (%) | Gain over initialization (pp) | Median step (s) | Peak allocated memory (GiB) |',
        '|---|---:|---:|---:|---:|']
    for row in summary:
        def show(key):return 'UNKNOWN'if row[key]is None else f"{row[key]:.4f}"
        lines.append('| '+row['variant']+' | '+' | '.join(show(key)for key in('top1_percent','gain_percentage_points','step_seconds','peak_GiB'))+' |')
    lines+=['','The common patch stem, every block MLP/norm1/norm2, final norm and predictor share initialization. Mixer architecture and parameter counts differ. Local and local-plus-far also share their near-factor and mixer-projection initialization. Local minus MLP assesses local mixing; local-plus-far minus local assesses adding the far branch and its extra parameters. The latter does not isolate hierarchy at matched parameter count. Equal exposure does not mean equal compute.',
        '', '| Exploratory final/gain contrast | Seed20261201 (pp) | Seed20261202 (pp) | Mean (pp) | Fragile95%pairedt interval,df1 (pp) |',
        '|---|---:|---:|---:|---:|']
    for name,stats in result['contrasts'].items():
        if name.endswith('u350'):
            lo,hi=stats['approximate_t_ci95_percentage_points'];deltas={row['seed']:row['delta_percentage_points']for row in stats['paired_seed_deltas']}
            lines.append(f"| {name} | {deltas[20261201]:+.4f} | {deltas[20261202]:+.4f} | {stats['mean_percentage_points']:+.4f} | [{lo:.4f},{hi:.4f}] |")
    lines+=['','| Variant | Encoder total | Encoder active | Trainable total | Trainable active | Unused norm1 | Mean feature std | Effective rank |',
        '|---|---:|---:|---:|---:|---:|---:|---:|']
    for row in summary:lines.append(f"| {row['variant']} | {row['encoder_parameters']:.0f} | {row['encoder_active_parameters']:.0f} | {row['trainable_parameters']:.0f} | {row['trainable_active_parameters']:.0f} | {row['inactive_context_scaffold_parameters']:.0f} | {row['feature_std_mean']:.6f} | {row['effective_rank']:.3f} |")
    lines+=['','The MLP-only control retains1536unused norm1 parameters solely for a common scaffold. Total counts include them; active counts exclude these known unused parameters and are not per-batch gradient counts. Feature diagnostics use the same predeclared subset before probe standardization; finite nonconstant features do not establish useful learning.',
        '', 'Both training seeds are shown as the replication units. With only two pairs, variance anddf1t intervals are unstable. All contrasts are exploratory and not multiplicity-adjusted. Intervals containing zero establish neither equivalence nor absence of an effect. Development images are reused; no new-dataset claim follows.',
        '', 'Step timing includes ordinary preprocessing and full JEPA training compute; it excludes probe/checkpoint overhead. Peak memory is allocated CUDA memory, excluding reserved cache and CUDA context. Counts, immutable states, caches and reconstructed predictions support every reported value.',
        '', 'Resume: Run All in Notebook09 with the same run ID and unchanged sources before the original deadline. An exhausted deadline never extends automatically.',
        '', '![Each seed before and after learning](stage_quality.png)',
        '', '![Local and far mechanism controls](stage_contrasts.png)',
        '', '![Measured training time and allocated memory](stage_compute.png)',
        '', 'Descriptive cost ratios: '+json.dumps(ratios,sort_keys=True)+'. No analytic scaling or universal speed guarantee is inferred.']
    path=artifact/'stage_results.md';path.write_text('\n'.join(lines)+'\n');atomic_json(artifact/'stage_summary_table.json',summary)
    return {'report':str(path.resolve()),'summary_table':str((artifact/'stage_summary_table.json').resolve()),'figures':figures,'cost_comparisons':ratios}

def _evaluate_stage_locked(root, artifact, plan, bundle, training, deadline):
    sources = _verify_plan_sources(root, plan)
    membership = _fixed_membership(root, bundle)
    jobs = _stage_jobs(plan, training)
    stage_role = plan.get("stage_role", plan["stage_id"])
    if stage_role not in STAGE_ROLES: raise ValueError("Unknown stage_role")
    if membership["pretraining_count"] != (65600 if stage_role == "data_volume" else 10100):
        raise ValueError("Pretraining count differs from the registered stage role")
    if int(plan["arms"][0]["config"]["image_size"]) != (256 if stage_role == "resolution" else 128):
        raise ValueError("Stage image resolution differs from declared role")
    identity = {"schema": "notebook05-stage-evaluation/v1", "plan": _canonical(plan),
        "source_hashes": sources, "evaluation_export_hash": _evaluation_export_hash(root),
        "dataset_fingerprint": bundle["dataset"].fingerprint, "membership": membership,
        "jobs": _canonical(jobs), "probe_selection": "fixed_final_epoch20", "official_test": "NOT_RUN"}
    identity_hash = _digest(identity)
    manifest_path = artifact/"stage_evaluation_manifest.json"
    if manifest_path.exists():
        manifest = _read(manifest_path)
        if manifest["identity_hash"] != identity_hash: raise ValueError("Stage evaluation identity changed; refusing stale reuse")
    else:
        manifest = {"identity_hash": identity_hash, "identity": identity, "completed": {}, "started_unix": time.time()}
        atomic_json(manifest_path, manifest)
    evaluations = {}
    for job in jobs:
        model_id, arm = job["model_id"], job["arm"]
        config = copy.deepcopy(arm["config"])
        summary_path = artifact/"evaluation"/(model_id+".json")
        _verified_record(job["state"])
        if model_id in manifest["completed"]:
            evaluations[model_id] = _load_saved_job(summary_path, manifest["completed"][model_id], identity_hash, job, membership)
            continue
        _deadline_check(deadline, 5)
        configure_environment(config)
        atomic_json(artifact/"evaluation_status.json", {"status":"RUNNING", "model_id":model_id,
            "completed_models":len(evaluations), "total_models":len(jobs), "updated_unix":time.time()})
        model = encoder = None
        try:
            _release_semantic(config["device"])
            model = build_model(config, arm["kind"]).cpu()
            parameter_counts={
                'context':sum(p.numel() for p in model.context.parameters()),
                'predictor':sum(p.numel() for p in model.predictor.parameters()),
                'teacher':sum(p.numel() for p in model.target.parameters()),
                'trainable':sum(p.numel() for p in model.parameters() if p.requires_grad),
                'inactive_context_scaffold':sum(p.numel() for name,p in model.context.named_parameters() if '.norm1.' in name) if arm['variant']=='mlp' else 0}
            parameter_counts['context_active']=parameter_counts['context']-parameter_counts['inactive_context_scaffold']
            parameter_counts['trainable_active']=parameter_counts['trainable']-parameter_counts['inactive_context_scaffold']
            state = torch.load(job["state"]["path"], map_location="cpu", weights_only=job["updates"] == 0)
            if job["updates"] == 0:
                model.target.load_state_dict(state)
            else:
                if int(state["state"]["successful_updates"]) != job["updates"] or state["config"] != arm["config"]:
                    raise ValueError("Checkpoint milestone/config differs from the registered schedule")
                model.load_state_dict(state["model"])
            del state
            encoder = model.target
            model = None
            common = {"dataset_revision":bundle["dataset"].fingerprint, "checkpoint_hash":encoder_state_hash(encoder),
                "encoder_kind":getattr(encoder,"kind",arm["kind"]), "preprocessing":preprocessing_descriptor(config),
                "pooling":"mean_final_layernorm_patches",
                "normalization":"encoder final affine LayerNorm then patch mean; train-only per-feature probe standardization"}
            caches, extraction = {}, []
            for split, ids in (("train", membership["probe_train_ids"]), ("development", membership["development_ids"])):
                _deadline_check(deadline, 2)
                dataset = _DeadlineDataset(EvaluationDataset(bundle["dataset"], ids, config), deadline)
                print(stage_role, model_id, split, len(ids), "fixed labeled feature IDs", flush=True)
                cache = extract_features(encoder, dataset, artifact/"features"/model_id,
                    {**common,"sample_ids":ids}, batch_size=int(config["evaluation"]["batch_size"]), device=config["device"])
                caches[split] = cache
                extraction.append({"split":split, "cache_reused":cache["reused"], **cache["performance"]})
            encoder.cpu()
            _release_semantic(config["device"])
            diagnostics = _cached_representation_diagnostics(caches["train"], caches["development"], bundle.get("diagnostic_ids", []))
            result = _fit_fixed_probe(caches["train"], caches["development"],
                root/"checkpoints"/plan["run_id"]/"stages"/plan["stage_id"]/"probes", config, model_id, encoder, deadline, plan)
            saved = {"identity_hash":identity_hash, "model_id":model_id, "arm_id":arm["id"], "kind":arm["kind"], "variant":arm["variant"],
                "seed":int(arm["seed"]), "stage_role":stage_role, "updates":job["updates"], "control":job["control"],
                "source_encoder":job["state"], "feature_caches":{name:_state_record(cache["path"]) for name,cache in caches.items()},
                "probe":result, "parameter_counts":parameter_counts, "representation_diagnostics":diagnostics, "feature_extraction":extraction,
                "training_cost":job["training_cost"], "heldout_status":"NOT_RUN"}
            write_csv(artifact/"predictions"/(model_id+".csv"), result["predictions"])
            atomic_json(summary_path, saved)
            manifest["completed"][model_id] = _state_record(summary_path)
            atomic_json(manifest_path, manifest)
            evaluations[model_id] = saved
            print(stage_role, model_id, "fixed epoch20 top1", 100*result["summary"]["metrics"]["top1"], flush=True)
        finally:
            model = encoder = None
            _release_semantic(config["device"])
    _deadline_check(deadline)
    if len(evaluations) != len(jobs): raise AssertionError("Every registered initial/final probe is required")
    probe_inits = {x["probe"]["summary"]["initialization_hash"] for x in evaluations.values()}
    probe_budgets = {_digest(x["probe"]["summary"]["budget"]) for x in evaluations.values()}
    probe_orders = {_digest([(r["epoch"],r["split"],r["permutation_hash"]) for r in x["probe"]["metrics"]]) for x in evaluations.values()}
    if len(probe_inits) != 1 or len(probe_budgets) != 1 or len(probe_orders) != 1:
        raise AssertionError("Probe initialization/budget/order is not identical across stage comparisons")
    scientific = analyze_stage_evidence(evaluations)
    result = {"schema":"notebook05-stage-evaluation/v1", "status":"COMPLETE", "run_id":plan["run_id"],
        "stage_id":plan["stage_id"], "stage_role":stage_role, "identity_hash":identity_hash,
        "plan_hash":_digest(plan), "dataset_fingerprint":bundle["dataset"].fingerprint,
        "membership":membership, "evaluations":evaluations, "milestones":[0]+list(plan["milestones"]),
        "schedule_note":"Exactly350updates fixed before discarded feasibility calibration; initialized and final encoders use fixedepoch20probes.",
        "image_size":int(plan["arms"][0]["config"]["image_size"]), "pretraining_images":len(bundle["train_ids"]),
        "training_design":{"updates":int(plan["updates"]),"effective_batch":plan["arms"][0]["config"]["training"]["microbatch"]*plan["arms"][0]["config"]["training"]["accumulation_steps"],
            "lr":plan["arms"][0]["config"]["training"]["lr"],"ema_start":.99,
            "weight_decay":plan["arms"][0]["config"]["training"]["weight_decay"],
            "warmup_fraction":plan["arms"][0]["config"]["training"]["warmup_fraction"],
            "patch_size":plan["arms"][0]["config"]["model"]["patch_size"],
            "model":copy.deepcopy(plan["arms"][0]["config"]["model"]), "dtype":plan["arms"][0]["config"].get("dtype","float32") },
        "probe_selection":"fixed_final_epoch20", "labels_in_self_supervised_loss":False,
        "official_test":"NOT_RUN", "started_unix":manifest["started_unix"], "completed_unix":time.time(),
        "learning_gate":scientific["learning_gate"], "hypothesis_assessment":scientific["hypothesis_assessment"], "contrasts":scientific["contrasts"], "chart_rows":scientific["chart_rows"],
        "report_scope":"Two fresh paired seeds, four architectures,350updates,16initial/final probes. All contrasts exploratory; priorpilots excluded; developmentimages reused."}
    atomic_json(artifact/"stage_evaluation.json", result)
    result["presentation"] = write_stage_report(artifact, result)
    atomic_json(artifact/"stage_evaluation.json", result)
    atomic_json(artifact/"evaluation_status.json", {"status":"COMPLETE", "completed_models":len(jobs), "learning_gate":result["learning_gate"], "updated_unix":time.time()})
    return result

def verify_evaluation_smoke(root,artifact,plan,bundle,deadline_unix):
    """Validate real encoder/cache integration and the unchanged fixed20epoch probe path.

    Old07features are used only as validation inputs, never as09quality evidence.
    Fresh09teacher features use two real training and two real development images per architecture.
    """
    root,artifact=Path(root).resolve(),Path(artifact).resolve();artifact.relative_to(root)
    artifact.mkdir(parents=True,exist_ok=True)
    old_root=root/'artifacts/notebook_07/energy-encoder-001'
    old_summary=old_root/'evaluation_recovery_v1/evaluation/s20261011-energy-initial.json'
    old_plan_path=old_root/'stages/learning/plan.json'
    source_summary=_read(old_summary);old_plan=_read(old_plan_path)
    provenance={'summary':_state_record(old_summary),'plan':_state_record(old_plan_path),
        'teacher':source_summary['source_encoder'],'caches':source_summary['feature_caches']}
    identity={'schema':'notebook09-evaluation-smoke/v1','plan_hash':_digest(plan),
        'source_hashes':RUNNER.stage_definition_hashes(root),'validation_only_source07':provenance,
        'fresh_training_pixels':list(bundle['train_ids'][:2]),'fresh_development_pixels':list(bundle['dev_ids'][:2]),
        'scope':'Fourfreshencoderkind realfeature/tinyprobeintegration plus interrupted-resumed fullfinalepoch20probe on old07validationcaches; excludedfrom09scientificresults.'}
    record_path=artifact/'evaluation_smoke.json'
    if record_path.exists():
        record=_read(record_path)
        if record['identity']!=identity or record.get('status')!='PASS':raise ValueError('Evaluation smoke identity/status changed')
        for record_file in record['verified_outputs']:_verified_record(record_file)
        return record
    _deadline_check(deadline_unix,5)
    outputs=[];checks={}
    for variant in ('vit','mlp','local','local_far'):
        arm=next(a for a in plan['arms'] if a['variant']==variant)
        config=copy.deepcopy(arm['config']);configure_environment(config)
        models,initial=RUNNER.initialize_models(config,artifact/'fresh_initial_models'/variant)
        model=models[arm['kind']];del models
        encoder=model.target
        caches={}
        for split,ids in [('train',list(bundle['train_ids'][:2])),('development',list(bundle['dev_ids'][:2]))]:
            dataset=_DeadlineDataset(EvaluationDataset(bundle['dataset'],ids,config),deadline_unix)
            feature_identity={'dataset_revision':bundle['dataset'].fingerprint,'checkpoint_hash':encoder_state_hash(encoder),
                'encoder_kind':encoder.kind,'preprocessing':preprocessing_descriptor(config),
                'pooling':'mean_final_layernorm_patches',
                'normalization':'encoder final affine LayerNorm then patch mean; train-only per-feature probe standardization','sample_ids':ids}
            cache=extract_features(encoder,dataset,artifact/'fresh_initial_features'/variant,feature_identity,batch_size=2,device=config['device'])
            if cache['identity']['encoder_kind']!=variant or cache['features'].shape!=(2,192):raise AssertionError('Fresh encoder integration mismatch')
            caches[split]=cache;outputs.append(_state_record(cache['path']))
        tiny_config=copy.deepcopy(config);tiny_config['probe'].update(epochs=20,batch_size=256)
        tiny_root=root/'checkpoints'/plan['run_id']/'validation'/'tiny_actual_kind_probes'/variant
        with _probe_checkpoint_metadata(plan):
            reference=fit_linear_probe(caches['train'],caches['development'],tiny_root,tiny_config,'reference',encoder=encoder)
            partial=fit_linear_probe(caches['train'],caches['development'],tiny_root,tiny_config,'resumed',encoder=encoder,stop_after_updates=1)
            resumed=fit_linear_probe(caches['train'],caches['development'],tiny_root,tiny_config,'resumed',encoder=encoder)
        reference_state=_load_probe_checkpoint(tiny_root/'reference',reference['selected_probe']['identity'])
        resumed_state=_load_probe_checkpoint(tiny_root/'resumed',resumed['selected_probe']['identity'])
        if reference_state['successful_updates']!=20 or resumed_state['successful_updates']!=20 or resumed_state['completed_epochs']!=20:
            raise AssertionError('Tiny real-kind epoch20 probe did not finish')
        for key,tensor in reference_state['classifier'].items():
            if not torch.equal(tensor,resumed_state['classifier'][key]):raise AssertionError('Actual-kind probe resume changed the final classifier')
        x=standardize(caches['development']['features'],resumed_state['standardization'])
        reconstructed=F.linear(x,resumed_state['classifier']['weight'],resumed_state['classifier']['bias'])
        if not torch.isfinite(reconstructed).all():raise AssertionError('Nonfinite tiny real-kind probe reconstruction')
        for name in ('reference','resumed'):outputs.append(_state_record(tiny_root/name/'probe_manifest.json'))
        checks[variant]={'status':'PASS','actual_encoder_kind':encoder.kind,'train_examples':2,'development_examples':2,
            'tiny_probe_completed_epochs':20,'tiny_probe_updates':20,'interrupted_resumed_classifier_bitwise_equal':True,
            'final_logits_reconstructed':True,'quality_result_use':'FORBIDDEN_validation_only'}
        model=encoder=cache=caches=reference=resumed=reference_state=resumed_state=None;_release_semantic(config['device'])
    _deadline_check(deadline_unix,5)
    old_arm=next(a for a in old_plan['arms'] if a['id']==source_summary['arm_id'])
    probe_config=copy.deepcopy(old_arm['config']);configure_environment(probe_config)
    with RUNNER.isolated_rng(int(old_arm['seed'])):
        fixture_runner=RUNNER.load_tagged(root,'experiments/07_energy/runner.ipynb','fmm09_probe_fixture_runner')
        model=fixture_runner.build_model(probe_config,old_arm['kind']).cpu()
    model.target.load_state_dict(torch.load(_verified_record(source_summary['source_encoder']),map_location='cpu',weights_only=True))
    encoder=model.target;caches={}
    for split,record in source_summary['feature_caches'].items():
        path=_verified_record(record);metadata=_read(path.with_suffix('.json'))
        if metadata['sha256']!=digest_file(path):raise ValueError('Validation-only07cache metadata mismatch')
        cache=torch.load(path,map_location='cpu',weights_only=True);validate_feature_cache(cache,metadata['identity'])
        if cache['identity']['encoder_kind']!='energy' or cache['identity']['checkpoint_hash']!=encoder_state_hash(encoder):
            raise ValueError('Validation-only07energy encoder/cache identity mismatch')
        caches[split]=cache
    checkpoint_root=root/'checkpoints'/plan['run_id']/'validation'/'evaluation_smoke_probes'
    model_id='validation-only-07-energy-initial'
    with _probe_checkpoint_metadata(plan):
        partial=fit_linear_probe(caches['train'],caches['development'],checkpoint_root,probe_config,model_id,encoder=encoder,stop_after_updates=17)
    if partial.get('status')!='INTERRUPTED' and not (checkpoint_root/model_id/'probe_manifest.json').exists():
        raise AssertionError('Probe lifecycle failed before resumable checkpoint')
    fixed=_fit_fixed_probe(caches['train'],caches['development'],checkpoint_root,probe_config,model_id,encoder,deadline_unix,plan)
    if fixed['reconciliation']['status']!='PASS' or fixed['summary']['selected_epoch']!=20 or fixed['summary']['successful_probe_updates']!=800:
        raise AssertionError('Fixed epoch20 probe did not reconstruct')
    for metric in ('top1','top5','loss'):
        if not math.isclose(fixed['summary']['metrics'][metric],source_summary['probe']['summary']['metrics'][metric],rel_tol=1e-6,abs_tol=1e-7):
            raise AssertionError('Interrupted/resumed validation probe differs from original fixed-probe result: '+metric)
    outputs.append(fixed['final_probe']);checks['fixed_probe']={'status':'PASS','selected_epoch':20,'successful_updates':800,
        'controlled_interruption_at':17,'predictions_reconstructed':fixed['reconciliation']['predictions_reconstructed'],
        'matched_original07_fixed_probe':True,'quality_result_use':'FORBIDDEN_validation_only'}
    model=encoder=None;_release_semantic(probe_config['device'])
    record={'status':'PASS','identity':identity,'checks':checks,'verified_outputs':outputs,'completed_unix':time.time()}
    atomic_json(record_path,record)
    return record

for _name in ('build_model','_evaluation_export_hash','_verify_plan_sources','_stage_jobs','_seed_statistics',
              'analyze_stage_evidence','write_stage_report','_evaluate_stage_locked'):
    setattr(_BASE,_name,globals()[_name])
_BASE.RESUME_NOTEBOOK=RUNNER.RESUME_NOTEBOOK
_BASE.RESUME_CELL=RUNNER.RESUME_CELL
evaluate_stage=_BASE.evaluate_stage
